In [ ]:
import torch
from datasets import load_dataset
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import torchvision
from torchvision import models
import torchvision.transforms as tr
from torch.cuda.amp import autocast
from sklearn.model_selection import train_test_split
import torchvision.io
import os
import time
!pip install mlflow
import mlflow
import numpy as np
from accelerate import Accelerator

In [ ]:

accelerator = Accelerator(mixed_precision="fp16")
device = accelerator.device
print(device)

cuda


In [ ]:
dataset = load_dataset("ethz/food101")

In [ ]:
test_val_splits = dataset['train'].train_test_split(test_size=0.2, seed=42)

In [ ]:
train_dataset = test_val_splits['train']
validation_dataset = test_val_splits['test']
test_dataset = dataset['validation']
train_dataset = train_dataset.select(range(10000))
validation_dataset = validation_dataset.select(range(2000))
test_dataset = test_dataset.select(range(2000))

In [ ]:
train_transform = tr.Compose([
    tr.Resize(256),
    tr.RandomResizedCrop(224),
    tr.RandomHorizontalFlip(0.5),
    tr.ToTensor(),
    tr.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

test_transform = tr.Compose([
    tr.Resize(256),
    tr.CenterCrop(224),
    tr.ToTensor(),
    tr.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

In [ ]:
def train_transforms(examples):
    examples["pixel_values"] = [train_transform(image.convert("RGB")) for image in examples["image"]]
    return examples


def test_transforms(examples):
    examples["pixel_values"] = [test_transform(image.convert("RGB")) for image in examples["image"]]
    return examples


train_dataset.set_transform(train_transforms)
validation_dataset.set_transform(test_transforms)
test_dataset.set_transform(test_transforms)

In [ ]:
sample = train_dataset[0]
print(sample.keys())


dict_keys(['image', 'label', 'pixel_values'])


In [ ]:
def collate_fn(batch):
    pixel_values = torch.stack([
        item["pixel_values"]
        for item in batch
    ])

    labels = torch.tensor([
        item["label"]
        for item in batch
    ])

    return {
        "pixel_values": pixel_values,
        "label": labels
    }
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
validation_loader = DataLoader(validation_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
  param.requires_grad = True


model.fc = nn.Linear(512, 101)
model = model.to(device)

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr = 1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, validation_loader
)

In [ ]:
patience = 2
patience_counter = 0
best_val_score = float("inf")


if accelerator.is_main_process:
    mlflow.set_experiment("ResNet18 Food101")
    mlflow.set_tracking_uri("https://attempt-evict-comment.ngrok-free.dev")
    mlflow.start_run(run_name="ResNet18_DualGPU")
    mlflow.log_params({
        "model": "ResNet18",
        "batch_size": 64,
        "learning_rate": 1e-4,
        "epochs": 10,
        "num_gpus": accelerator.num_processes
    })


for epoch in range(15):
    accelerator.print(f"\n===== STARTING EPOCH {epoch + 1}/10 =====")
    start_epoch = time.time()

    # --- TRAINING ---
    model.train()
    train_start = time.time()
    train_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)

        inputs = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)


        with accelerator.autocast():
            preds = model(inputs)
            loss = criterion(preds, labels)

        accelerator.backward(loss)

        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_loss += loss.item()

    train_time = time.time() - train_start


    model.eval()
    validation_start = time.time()
    all_preds = []
    all_labels = []
    test_loss = 0.0

    with torch.no_grad():
        for batch in validation_loader:

            with accelerator.autocast():
                preds = model(batch["pixel_values"].to(device, non_blocking=True))
                loss = criterion(preds, batch["label"].to(device, non_blocking=True))
            test_loss += loss.item()

            pred = torch.argmax(preds, dim=1)


            all_preds.append(pred)
            all_labels.append(batch["label"])

    validation_time = time.time() - validation_start


    all_preds_tensor = torch.cat(all_preds)
    all_labels_tensor = torch.cat(all_labels)


    gathered_preds = accelerator.gather_for_metrics(all_preds_tensor).cpu().numpy()
    gathered_labels = accelerator.gather_for_metrics(all_labels_tensor).cpu().numpy()

    avg_test_loss = test_loss / len(validation_loader)
    avg_train_loss = train_loss / len(train_loader)

    accuracy = accuracy_score(gathered_labels, gathered_preds)
    f1 = f1_score(gathered_labels, gathered_preds, average="weighted")

    scheduler.step(avg_test_loss)


    if avg_test_loss < best_val_score:
        best_val_score = avg_test_loss
        patience_counter = 0
        if accelerator.is_main_process:
            unwrapped_model = accelerator.unwrap_model(model)
            torch.save(unwrapped_model.state_dict(), "resnet18_food101_best.pt")
    else:
        patience_counter += 1

    epoch_time = time.time() - start_epoch


    accelerator.print(f'Epoch {epoch + 1} train loss {avg_train_loss:.4f} test loss {avg_test_loss:.4f} accuracy {accuracy:.4f} f1 score {f1:.4f}')
    accelerator.print(f"Train time: {train_time:.2f}s | Val time: {validation_time:.2f}s | Epoch time: {epoch_time:.2f}s")

    if accelerator.is_main_process:
        mlflow.log_metrics({
            "train_loss": avg_train_loss,
            "val_loss": avg_test_loss,
            "accuracy": accuracy,
            "f1_score": f1
        }, step=epoch)

    if patience_counter == patience:
        accelerator.print(f"Early stopping was activated on {epoch + 1}! End.")
        break

if accelerator.is_main_process:
    mlflow.end_run()


===== STARTING EPOCH 1/10 =====
Epoch 1 train loss 3.9773 test loss 3.1655 accuracy 0.2940 f1 score 0.2538
Train time: 34.01s | Val time: 5.85s | Epoch time: 39.97s

===== STARTING EPOCH 2/10 =====
Epoch 2 train loss 2.8718 test loss 2.5036 accuracy 0.4260 f1 score 0.4040
Train time: 34.38s | Val time: 6.53s | Epoch time: 41.03s

===== STARTING EPOCH 3/10 =====
Epoch 3 train loss 2.3638 test loss 2.1489 accuracy 0.4940 f1 score 0.4852
Train time: 34.27s | Val time: 5.85s | Epoch time: 40.23s

===== STARTING EPOCH 4/10 =====
Epoch 4 train loss 2.0570 test loss 1.9863 accuracy 0.5195 f1 score 0.5218
Train time: 34.13s | Val time: 5.79s | Epoch time: 40.03s

===== STARTING EPOCH 5/10 =====
Epoch 5 train loss 1.8120 test loss 1.8827 accuracy 0.5490 f1 score 0.5468
Train time: 33.88s | Val time: 5.90s | Epoch time: 39.89s

===== STARTING EPOCH 6/10 =====
Epoch 6 train loss 1.6378 test loss 1.8186 accuracy 0.5575 f1 score 0.5503
Train time: 33.77s | Val time: 5.89s | Epoch time: 39.77s

===

In [ ]:
mlflow.end_run()